# OVCA leave-one-cell-type-out: `cellina` vs a labels baseline, on **WSI-predicted domains**

A port of [`03_loo_eval.ipynb`](03_loo_eval.ipynb) from the CRC (`crc_232`) CosMx dataset to the
**OVCA** Xenium Prime ovarian-cancer slide, with one substantive change: the spatial domain that
defines the counterfactual is the one **predicted from H&E** by
[`05_roi_embedding_classifiers.ipynb`](05_roi_embedding_classifiers.ipynb), not the pathologist's
ground-truth annotation.

The experiment keeps 03's structure: hold out one cell type in one domain, never showing that
combination during training, then ask each model to predict expression for control cells placed
counterfactually into that domain, and score `cellina` against the no-context
`wsi.LabelsBaseline` floor — split into **OOD** (the held-out type) and **ID** (everything else).

| | CRC (`03_loo_eval`) | OVCA (this notebook) |
|---|---|---|
| control -> target | `REF` -> `CRC` | `Smooth muscle` -> `Tumor` |
| held-out cell type | `Myeloid` | `Stromal` |
| cells | 111,989 | 14,170 |
| genes | 2,000 HVGs | 5,101-gene Xenium panel, **no HVG subsetting** |
| domain driving the counterfactual | ground truth | **`pred_domain` (from H&E)** |

`Smooth muscle -> Tumor` holding out `Stromal` is the OVCA analogue of CRC's myeloid-in-cancer
setup: quiescent stroma reprogramming into cancer-associated fibroblasts. It was chosen over the
alternatives on power — `Stromal` has 922 held-out cells in `Tumor` and 2,953 seen elsewhere,
where e.g. `Epithelial`-in-`Tumor` would have left only 66.

## Where predicted vs. ground-truth domain is used

Mixing these up is the easy way to silently invalidate the experiment, so, explicitly:

- **Ground-truth `domain`** defines the experiment: it is registered in `setup_anndata` and drives
  `split_indices`, so "never seen during training" means what it says.
- **Predicted `pred_domain`** defines the counterfactual: which cells form the control arm, which
  form the target arm, and which are eligible donors for the `Tumor` microenvironment.
- **`cell_type` is always ground truth.** Only the domain is swapped.

## ⚠️ Read this before interpreting the numbers

**`pred_domain` agrees with the ground-truth `domain` for 99.3% of these cells, so this run is
close to a no-op relative to just using the labels.** Two reasons, both worth keeping in view:

1. **The predictions are mostly in-sample.** ~96% of these cells (`split_05 == "train"`) sit in
   ROIs the 05 logistic regression was *trained* on, where it scores **100.0%** by memorisation.
   Only the `split_05 == "test"` cells (3,192) are genuine out-of-sample predictions, and there it
   scores 96.9%. So this does **not** demonstrate domain inference from H&E in any deployment
   sense — for most cells it is feeding memorised labels back in.
2. **Consequently the arms barely move.** Substituting `pred_domain` changes the control arm by 22
   cells and the target arm by ~46. Expect metrics close to a ground-truth-domain run.

The honest fix is out-of-fold predictions (refit the logreg K times over the 36 ROIs so every cell
is predicted by a model that never saw its ROI); the informative comparison is then the *delta*
against a ground-truth-domain reference arm. Both were considered and deliberately deferred —
this notebook is the as-is baseline. The diagnostic cell in section A prints the live agreement
table so the caveat travels with the results rather than living only in this markdown.

## Environments (this notebook needs two)

`OVCA.zarr` is **zarr v3** (written by `lazyslide-env`), which `cellina_edge`'s zarr 2.18.3 cannot
read. And an h5ad handoff does not work either: anndata 0.12 writes Arrow-backed strings that
anndata 0.10.8 rejects (`nullable-string-array`). So the join happens in a prep script and the
result crosses over as a version-neutral `.npz`/`.csv`/`.npy` bundle:

```bash
# once, in lazyslide-env -> writes data/ovca_loo/
/g/stegle/spiljak/programs/miniforge3/envs/lazyslide-env/bin/python scripts/prepare_ovca_loo_data.py
```

Then run **this notebook top to bottom in the `cellina_edge` kernel** (A -> B -> C).

| Section | Produces |
|---|---|
| **A. Shared setup + cellina** | `preds_ovca/observed.h5ad`, `preds_ovca/cellina_pred.h5ad` |
| **B. Labels baseline** | `preds_ovca/labels_pred.h5ad` |
| **C. Compare (OOD + ID)** | metric tables + plot |

In [ ]:
import os
import sys

import numpy as np
import pandas as pd

# ../scripts for the LOO plumbing (eval/metrics); ../src for the `wsi` package (LabelsBaseline),
# which is not pip-installed into cellina_edge.
sys.path.append(os.path.abspath("../scripts"))
sys.path.append(os.path.abspath("../src"))

from utils import set_seed
from eval import (
    load_ovca_loo_bundle,
    preprocess_ovca,
    preprocess_spatial_features,
    split_indices,
    make_comparison_adata,
    run_loo_eval,
    COUNTS_PER_K,
)
from configs.adata_ovca_config import ADATA_ARGS

set_seed(0)

# --- experiment config (see configs/adata_ovca_config.py) -------------------
# Built by scripts/prepare_ovca_loo_data.py under lazyslide-env -- see the header.
BUNDLE_DIR = os.path.abspath("../data/ovca_loo")

LABELS_KEY = ADATA_ARGS["labels_key"]             # "cell_type"      (always ground truth)
DOMAINS_KEY = ADATA_ARGS["domains_key"]           # "domain"         ground truth -> training + split
PRED_DOMAINS_KEY = ADATA_ARGS["pred_domains_key"] # "pred_domain"    from H&E     -> counterfactual arms

HOLDOUT_CT = ADATA_ARGS["holdout_celltype"]       # "Stromal"
CONTROL_LABEL = ADATA_ARGS["control_domains"][0]  # "Smooth muscle"  (quiescent stroma)
HOLDOUT_LABEL = ADATA_ARGS["holdout_domains"][0]  # "Tumor"          (held out for Stromal)

# Arm label on the comparison AnnDatas. Named distinctly from DOMAINS_KEY as a reminder that the
# arms are cut by PREDICTED domain, not by the ground-truth column of the same name.
PERT_COL = "domain_arm"

PRED_DIR = "preds_ovca"   # kept separate from 03's `preds/` so the CRC run is not clobbered
os.makedirs(PRED_DIR, exist_ok=True)

print(f"{CONTROL_LABEL!r} -> {HOLDOUT_LABEL!r}, holding out {HOLDOUT_CT!r}")
print(f"arms cut by {PRED_DOMAINS_KEY!r}; training/split by {DOMAINS_KEY!r}")

## A. Shared setup + cellina  *(`cellina_edge` kernel)*

In [ ]:
# Load the prepared bundle (raw counts + micron centroids + true/predicted domain), then the
# shared QC. Unlike preprocess_crc there is no label remap or coord assembly to do -- the bundle
# arrives clean. n_top_genes=None: the Xenium panel is already curated, so no HVG subsetting.
adata = load_ovca_loo_bundle(BUNDLE_DIR)
adata = preprocess_ovca(
    adata,
    n_top_genes=ADATA_ARGS["n_top_genes"],
    labels_key=LABELS_KEY,
    domains_key=DOMAINS_KEY,
    pred_domains_key=PRED_DOMAINS_KEY,
)
adata

In [ ]:
# Diagnostic: how much does the WSI-predicted domain actually differ from ground truth?
# This is the caveat from the header, measured live -- if `pred_equals_true` is ~1.0 on the
# split_05=="train" rows, those "predictions" are memorised, not inferred.
agree = adata.obs[DOMAINS_KEY].astype(str) == adata.obs[PRED_DOMAINS_KEY].astype(str)
print(f"pred_domain == domain for {agree.mean():.2%} of {adata.n_obs} cells\n")
print(
    adata.obs.assign(agree=agree)
    .groupby("split_05")["agree"]
    .agg(n_cells="size", pred_equals_true="mean")
    .to_string()
)

print("\narm sizes, ground truth vs predicted:")
rows = []
for arm in (CONTROL_LABEL, HOLDOUT_LABEL):
    t = (adata.obs[DOMAINS_KEY].astype(str) == arm).values
    p = (adata.obs[PRED_DOMAINS_KEY].astype(str) == arm).values
    rows.append({"arm": arm, "n_true": int(t.sum()), "n_pred": int(p.sum()),
                 "in_both": int((t & p).sum()), "pred_only": int((p & ~t).sum()),
                 "true_only": int((t & ~p).sum())})
print(pd.DataFrame(rows).to_string(index=False))

print(f"\n{LABELS_KEY} x {PRED_DOMAINS_KEY}:")
print(pd.crosstab(adata.obs[LABELS_KEY], adata.obs[PRED_DOMAINS_KEY]))

In [ ]:
# Leave-one-cell-type-out split: test = Stromal cells in the Tumor domain.
# NOTE the split is cut by GROUND-TRUTH domain on purpose -- it defines what the model was
# actually never trained on, so the OOD claim stays meaningful even though the arms below are
# cut by the predicted domain.
train_idx, val_idx, test_idx = split_indices(
    adata, HOLDOUT_CT, labels_key=LABELS_KEY, domains_key=DOMAINS_KEY,
    holdout_domains=[HOLDOUT_LABEL], seed=0,
)
print(f"train={len(train_idx)}  val={len(val_idx)}  test={len(test_idx)}   "
      f"(test = {HOLDOUT_CT} cells in {HOLDOUT_LABEL}, by ground-truth domain)")

In [ ]:
# Spatial neighbour graph + aggregated features (held-out cells masked to avoid leakage).
# bandwidth is passed in MICRONS: OVCA's obsm["spatial"] is already in microns, whereas CRC's
# coords are pixels (hence its 100/step_size_px conversion). cellina's spatial_neighbors takes
# bandwidth in the units of obsm["spatial"], so passing 100.0 here means a 100um neighbourhood.
adata = preprocess_spatial_features(
    adata,
    bandwidth=ADATA_ARGS["bandwidth_um"],
    n_neighbors=ADATA_ARGS["n_neighbors"],
    test_indices=test_idx,
)

In [ ]:
# Control = ALL predicted-Smooth-muscle cells; observed target = ALL predicted-Tumor cells.
# We run the counterfactual over every cell type in one pass, then split scoring (section C) into:
#   - OOD: the held-out cell type (Stromal) -> (Stromal, Tumor) was never seen in training
#   - ID : the held-in cell types           -> (cell type, Tumor) was seen in training
# Both arms are cut by PREDICTED domain (the point of this notebook).
is_holdout_ct = adata.obs[LABELS_KEY].astype(str) == HOLDOUT_CT
mask_control = (adata.obs[PRED_DOMAINS_KEY].astype(str) == CONTROL_LABEL).values
mask_target = (adata.obs[PRED_DOMAINS_KEY].astype(str) == HOLDOUT_LABEL).values
idx_control = np.where(mask_control)[0]
idx_target = np.where(mask_target)[0]

# Cell-type labels per arm, so we can slice OOD vs ID after scoring. The counterfactual moves
# control cells into the Tumor microenvironment but keeps each cell's own type, so the predicted
# arm carries the CONTROL cells' types.
ct_control = adata.obs[LABELS_KEY].astype(str).values[mask_control]
ct_target = adata.obs[LABELS_KEY].astype(str).values[mask_target]

# Donor pool for the Tumor microenvironment: observed neighbours of the (predicted) Tumor cells,
# excluding the held-out cell type so held-out cells never leak in as donors.
conn = adata.obsp["spatial_connectivities_orig"]
neighbor_indices = np.unique(conn[idx_target].nonzero()[1])
neighbor_indices = neighbor_indices[~is_holdout_ct.values[neighbor_indices]]
print(f"control(pred {CONTROL_LABEL})={idx_control.size}  target(pred {HOLDOUT_LABEL})={idx_target.size}  "
      f"donor-neighbours={neighbor_indices.size}  OOD-celltype={HOLDOUT_CT!r}")
print(f"  of which {HOLDOUT_CT} in the target arm: {int((ct_target == HOLDOUT_CT).sum())}")

In [ ]:
# Observed pair (ground truth expression): real control vs real target counts across all cell
# types, with the cell-type label attached so section C can slice OOD from ID.
observed = make_comparison_adata(
    adata.layers["counts"][mask_control, :],
    adata.layers["counts"][mask_target, :],
    adata.var_names, control_label=CONTROL_LABEL, perturbed_label=HOLDOUT_LABEL,
    pert_col=PERT_COL,
)
observed.obs[LABELS_KEY] = np.concatenate([ct_control, ct_target])
observed.write(os.path.join(PRED_DIR, "observed.h5ad"))
observed

In [ ]:
# --- Train cellina (same args as the CRC run; configs/cellina_config.py) ---
from cellina import Cellina
from configs.cellina_config import (
    MODEL_ARGS as CELLINA_MODEL_ARGS,
    TRAIN_ARGS as CELLINA_TRAIN_ARGS,
    PLAN_KWARGS as CELLINA_PLAN_KWARGS,
)

# domains_key = ground-truth domain: the model is conditioned on / adversarially forgets the real
# domain, and the split above guarantees it never sees (Stromal, Tumor).
Cellina.setup_anndata(
    adata, batch_key=ADATA_ARGS["batch_key"], labels_key=LABELS_KEY, domains_key=DOMAINS_KEY,
    spatial_obsm_key="spatial_x", layer="counts",
)
cellina_model = Cellina(adata, **CELLINA_MODEL_ARGS)

train_args = dict(CELLINA_TRAIN_ARGS)
train_args["datasplitter_kwargs"] = {"external_indexing": [train_idx, val_idx, test_idx]}
cellina_model.train(**train_args, plan_kwargs=CELLINA_PLAN_KWARGS)

In [ ]:
# Edge-perturbation counterfactual over ALL control cells: rewire each into the (predicted) Tumor
# microenvironment (one pass; sliced into OOD/ID at scoring time).
cellina_cf = cellina_model.get_counterfactual_expression(
    indices=idx_control, batch_size=2048, seed=0,
    neighbour_indices=neighbor_indices, precomputed=False,
)
# cellina's own reconstruction of the control cells (the predicted control arm).
cellina_ctrl = cellina_model.get_normalized_expression(
    adata[mask_control], library_size=COUNTS_PER_K, batch_size=2048,
)

cellina_pred = make_comparison_adata(
    cellina_ctrl, cellina_cf, adata.var_names,
    control_label=CONTROL_LABEL, perturbed_label=HOLDOUT_LABEL, pert_col=PERT_COL,
)
# Both arms are the control cells (reconstruction + their counterfactual).
cellina_pred.obs[LABELS_KEY] = np.concatenate([ct_control, ct_control])
cellina_pred.write(os.path.join(PRED_DIR, "cellina_pred.h5ad"))
cellina_pred

## B. Labels baseline  *(same `cellina_edge` kernel)*

The labels baseline reuses the same in-memory `adata` and splits from section A. It ignores
expression and spatial context entirely, predicting from the (cell-type, domain) labels alone —
one-hot encoded and decoded through an scvi `DecoderSCVI` with an NB likelihood and observed
library size. It is the no-context floor `cellina` should beat, especially OOD.

Its counterfactual is a label swap rather than a spatial rewire: take the control cells, relabel
their domain to `Tumor`, decode. Note the relabel target is a category of the **ground-truth**
`domain` column (the one registered in `setup_anndata`), while the *cells being relabelled* are
still selected by predicted domain — so the arms stay identical to cellina's and the section-C
comparison is apples-to-apples.

In [ ]:
# Train the labels baseline on the same split (reuses adata / splits from section A).
from wsi.labels_baseline import LabelsBaseline
from configs.labels_config import (
    MODEL_ARGS as LABELS_MODEL_ARGS,
    TRAIN_ARGS as LABELS_TRAIN_ARGS,
    PLAN_KWARGS as LABELS_PLAN_KWARGS,
)

LabelsBaseline.setup_anndata(
    adata, labels_key=LABELS_KEY, domains_key=DOMAINS_KEY,
    batch_key=ADATA_ARGS["batch_key"], layer="counts",
)
labels_model = LabelsBaseline(adata, **LABELS_MODEL_ARGS)

train_args = dict(LABELS_TRAIN_ARGS)
train_args["datasplitter_kwargs"] = {"external_indexing": [train_idx, val_idx, test_idx]}
labels_model.train(**train_args, plan_kwargs=LABELS_PLAN_KWARGS)

In [ ]:
# Counterfactual over ALL control cells -> Tumor (one pass), plus control reconstruction.
labels_cf = labels_model.get_counterfactual_expression(
    idx_control, target_domain=HOLDOUT_LABEL, batch_size=2048,
)  # observed (latent) library size; each cell keeps its own cell-type label
labels_ctrl = labels_model.get_normalized_expression(
    adata[mask_control], library_size=COUNTS_PER_K, batch_size=2048,
)

labels_pred = make_comparison_adata(
    labels_ctrl, labels_cf, adata.var_names,
    control_label=CONTROL_LABEL, perturbed_label=HOLDOUT_LABEL, pert_col=PERT_COL,
)
labels_pred.obs[LABELS_KEY] = np.concatenate([ct_control, ct_control])
labels_pred.write(os.path.join(PRED_DIR, "labels_pred.h5ad"))
labels_pred

## C. Compare — OOD + ID

Load the cached predictions and score each model with `compare_adatas` on the top-50
observed-effect genes, **split by regime**:

- **OOD** — cells of the held-out type (`Stromal`): predicting `(Stromal, Tumor)`, never seen.
- **ID** — cells of the held-in types: `(cell type, Tumor)` combinations seen in training.

Both regimes reuse the *same* one-pass predictions from sections A/B; we simply slice the
observed/predicted AnnDatas by the attached `cell_type`. `run_loo_eval` uses
`predicted_logfc_baseline="observed_control"`, so — as in the cellina tutorial — both the
ground-truth and predicted logFC are measured against the *same observed* control population
(within each regime's cell set).

Reminder from the header: the arms here are cut by `pred_domain`, which matches ground truth for
99.3% of cells, so these numbers should land close to a ground-truth-domain run. That closeness is
the expected outcome, not a validation of H&E-derived domain inference.

In [ ]:
import os, sys
import numpy as np
import pandas as pd
import scanpy as sc

sys.path.append(os.path.abspath("../scripts"))
from eval import run_loo_eval
from metrics import get_perturbation_effects

PRED_DIR = "preds_ovca"
PERT_COL, LABELS_KEY = "domain_arm", "cell_type"
CONTROL_LABEL, HOLDOUT_LABEL, HOLDOUT_CT = "Smooth muscle", "Tumor", "Stromal"

observed = sc.read(os.path.join(PRED_DIR, "observed.h5ad"))
cellina_pred = sc.read(os.path.join(PRED_DIR, "cellina_pred.h5ad"))
labels_pred = sc.read(os.path.join(PRED_DIR, "labels_pred.h5ad"))

PREDS = [("cellina", cellina_pred), ("labels", labels_pred)]


def score(mask_fn):
    \"\"\"Score each model's control -> Tumor prediction on the cells selected by mask_fn.\"\"\"
    res = []
    for name, pred in PREDS:
        o = observed[mask_fn(observed)].copy()
        p = pred[mask_fn(pred)].copy()
        row = run_loo_eval(o, p, pert_col=PERT_COL, control_label=CONTROL_LABEL).iloc[0]
        res.append(row.rename(name))
    return pd.DataFrame(res)


# OOD = held-out cell type (Stromal, never seen in Tumor); ID = held-in types (seen in Tumor).
comparison_ood = score(lambda a: (a.obs[LABELS_KEY].astype(str) == HOLDOUT_CT).values)
comparison_id = score(lambda a: (a.obs[LABELS_KEY].astype(str) != HOLDOUT_CT).values)

comparison = pd.concat({"OOD": comparison_ood, "ID": comparison_id}, names=["regime", "model"])
comparison

In [ ]:
# Observed vs predicted logFC scatter (cellina) on the OOD held-out Stromal, top DE genes.
import matplotlib.pyplot as plt
import numpy as np

ood = lambda a: a[(a.obs[LABELS_KEY].astype(str) == HOLDOUT_CT).values].copy()
obs_eff, pred_eff = get_perturbation_effects(
    ood(observed), ood(cellina_pred), pert_col=PERT_COL, control_label=CONTROL_LABEL,
    mode="logfc", predicted_logfc_baseline="predicted_control",
)
o = obs_eff.loc[HOLDOUT_LABEL].values
p = pred_eff.loc[HOLDOUT_LABEL].values
deg = np.argsort(-np.abs(o))[:50]

fig, ax = plt.subplots(figsize=(5.5, 5.5))
ax.scatter(o, p, s=8, color="lightgray", alpha=0.4)
ax.scatter(o[deg], p[deg], s=30, color="#d62728")
lims = [min(o.min(), p.min()), max(o.max(), p.max())]
ax.plot(lims, lims, "k--", lw=1, alpha=0.6)
ax.set_xlabel(f"Observed logFC ({CONTROL_LABEL} -> {HOLDOUT_LABEL})")
ax.set_ylabel("Predicted logFC (cellina counterfactual)")
ax.set_title(f"cellina: observed vs predicted logFC (OOD {HOLDOUT_CT})")
plt.tight_layout()
plt.show()